In [1]:
# importing libraries
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np

In [2]:
skincare = pd.read_csv("dataset/skincare.csv", index_col = 0)

In [3]:
skincare.head()

,product_name,brand_name,loves_count,rating,reviews,ingredients,price_usd,highlights,secondary_category,tertiary_category,size (oz)
15,GENIUS Ultimate Anti-Aging Melting Cleanser,Algenist,9314,4.0569,334.0,"['C12-15 Alkyl Benzoate, Ethylhexyl Palmitate,...",38.0,NaN,Cleansers,Face Wash & Cleansers,5.00
16,Gentle Rejuvenating Cleanser,Algenist,7681,4.2689,264.0,"['Water, Sodium Cocoyl Isethionate, Glyceryl S...",28.0,NaN,Cleansers,Face Wash & Cleansers,4.00
17,Advanced Anti-Aging Repairing Oil,Algenist,10676,4.4531,245.0,"['Chlorella Protothecoides Oil, Cetearyl Ethyl...",82.0,NaN,Moisturizers,Face Oils,1.00
18,ALIVE Prebiotic Balancing Mask,Algenist,14367,4.3729,118.0,"['Glycerin, Water (Aqua, Eau), Sodium Cocoyl G...",38.0,"['Vegan', 'Hypoallergenic', 'Good for: Acne/Bl...",Masks,Face Masks,1.70
19,Balancing Cleanser,Alpha-H,3612,4.5455,77.0,"['Water, Ethylhexyl Palmitate, Aloe Barbadensi...",38.0,"['Best for Dry, Combo, Normal Skin', 'Good for...",Cleansers,Face Wash & Cleansers,6.25


In [4]:
skincare.columns

Index(['product_name', 'brand_name', 'loves_count', 'rating', 'reviews',
       'ingredients', 'price_usd', 'highlights', 'secondary_category',
       'tertiary_category', 'size (oz)'],
      dtype='object')

In [5]:
# columns where text mining can be conducted
text_cols = skincare.columns[skincare.dtypes == object]
text_cols

Index(['product_name', 'brand_name', 'ingredients', 'highlights',
       'secondary_category', 'tertiary_category'],
      dtype='object')

In [6]:
# mining ingredients column
import re 

# parsing ingredients into lists
def parse_ingredients(ingredient_entry):
    if pd.isna(ingredient_entry):
        return []
    
    if isinstance(ingredient_entry, list) and len(ingredient_entry) > 0:
        text = ingredient_entry[0] 
    else:
        text = str(ingredient_entry)
    ingredients = [i.strip() for i in text.split(',')]
    
    cleaned = []
    for ing in ingredients:
        ing = re.sub(r'\d+%', '', ing)
        ing = re.sub(r'\s*\([^)]*\)', '', ing)
        ing = re.sub(r'\s+', ' ', ing).strip().lower()
        ing = ing.strip("[]'\" ")
        ing = re.sub(r"[.,;:]+$", "", ing)
        
        if ing and len(ing) > 1: 
            cleaned.append(ing)
    
    return cleaned

# testing parsing function
sample_idx = 0
print(f"Raw input: {skincare['ingredients'].iloc[sample_idx]}")
print(f"Parsed output: {parse_ingredients(skincare['ingredients'].iloc[sample_idx])}")

# Apply parsing to all rows
skincare['ings_list'] = skincare['ingredients'].apply(parse_ingredients)

# Define ingredient categories
ingredient_categories = {
    'cleansers': ['lauryl', 'coco', 'glucoside', 'betaine', 'sulfate', 'cocamidopropyl'],
    'moisturizers': ['glycerin', 'hyaluronic', 'ceramide', 'squalane', 'shea', 'cetearyl', 'butyrospermum'],
    'exfoliants': ['salicylic', 'glycolic', 'lactic', 'mandelic', 'aha', 'bha', 'acid'],
    'actives': ['niacinamide', 'retinol', 'vitamin c', 'ascorbic', 'peptide', 'tocopherol'],
    'soothing': ['aloe', 'chamomile', 'panthenol', 'allantoin', 'bisabolol', 'chamomilla'],
    'preservatives': ['phenoxyethanol', 'paraben', 'sorbate', 'benzoate', 'ethylhexylglycerin'],
    'fragrance': ['fragrance', 'parfum', 'limonene', 'linalool', 'hexyl'],
    'oils': ['oil', 'triglyceride', 'ester', 'oleate', 'caprylic'],
    'water_based': ['aqua', 'water', 'eau']
}

# Create count columns (how many ingredients in each category)
for category, keywords in ingredient_categories.items():
    skincare[f'count_{category}'] = skincare['ings_list'].apply(
        lambda x: sum(1 for ing in x if any(k in ing for k in keywords))
    )

# Create binary has columns (whether category is present)
for category in ingredient_categories.keys():
    skincare[f'has_{category}'] = (skincare[f'count_{category}'] > 0).astype(int)

# check if any product has zero ingredients in all categories
count_cols = [col for col in skincare.columns if col.startswith('count_')]
print(f"\n✅ Products with no categorized ingredients: {(skincare[count_cols].sum(axis=1) == 0).sum()} out of {len(skincare)}")

Raw input: ['C12-15 Alkyl Benzoate, Ethylhexyl Palmitate, Caprylic/Capric Triglyceride, Glycerin, Water (Aqua), Sucrose Laurate, Olea Europaea (Olive) Fruit Oil, Persea Gratissima (Avocado) Oil, Chlorella Protothecoides Oil, Algae Exopolysaccharides, Prunus Amygdalus Dulcis (Sweet Almond) Oil, Vaccinium Myrtillus Seed Oil, Retinyl Palmitate, Tocopheryl Acetate, Caprylyl Glycol, Chlorphenesin, Phenoxyethanol, Fragrance (Parfum), Benzyl Benzoate, Chromium Oxide Greens (CI 77288).']
Parsed output: ['c12-15 alkyl benzoate', 'ethylhexyl palmitate', 'caprylic/capric triglyceride', 'glycerin', 'water', 'sucrose laurate', 'olea europaea fruit oil', 'persea gratissima oil', 'chlorella protothecoides oil', 'algae exopolysaccharides', 'prunus amygdalus dulcis oil', 'vaccinium myrtillus seed oil', 'retinyl palmitate', 'tocopheryl acetate', 'caprylyl glycol', 'chlorphenesin', 'phenoxyethanol', 'fragrance', 'benzyl benzoate', 'chromium oxide greens']

✅ Products with no categorized ingredients: 12 o

In [7]:
#no ingredients 
count_cols = [c for c in skincare.columns if c.startswith("count_")]
no_match = skincare[skincare[count_cols].sum(axis=1) == 0].copy()

no_match[["product_name", "brand_name", "secondary_category", "ings_list"]].head(12)

,product_name,brand_name,secondary_category,ings_list
354,Cryo-Recovery Lifting Face Mask with Acupressu...,Charlotte Tilbury,High Tech Tools,[]
1287,Original Skin Retexturizing Mask with Rose Clay,Origins,Masks,[]
1292,Out of Trouble 10 Minute Mask to Rescue Proble...,Origins,Masks,[]
1490,Vegan Makeup Remover and Cleansing Brush,SEPHORA COLLECTION,High Tech Tools,[]
1497,1 Minute Face Masks,SEPHORA COLLECTION,Masks,[]
1504,Face Mask Applicator,SEPHORA COLLECTION,Masks,[]
1513,Reusable 3-Piece Grey Face Mask Set,SEPHORA COLLECTION,Wellness,[]
1525,Shani Darden by Déesse PRO LED Light Mask,Shani Darden Skin Care,High Tech Tools,[]
1531,White Lucent Power Brightening Mask,Shiseido,Masks,[]
1573,Silk Sleepmask,Slip,Wellness,[]


In [8]:
skincare.columns

Index(['product_name', 'brand_name', 'loves_count', 'rating', 'reviews',
       'ingredients', 'price_usd', 'highlights', 'secondary_category',
       'tertiary_category', 'size (oz)', 'ings_list', 'count_cleansers',
       'count_moisturizers', 'count_exfoliants', 'count_actives',
       'count_soothing', 'count_preservatives', 'count_fragrance',
       'count_oils', 'count_water_based', 'has_cleansers', 'has_moisturizers',
       'has_exfoliants', 'has_actives', 'has_soothing', 'has_preservatives',
       'has_fragrance', 'has_oils', 'has_water_based'],
      dtype='object')

In [9]:
# mining highlights column 

# parsing highlights into one list 
def parse_highlights(highlight_entry):
    if pd.isna(highlight_entry):
        return []
    
    if isinstance(highlight_entry, list):
        return [str(h).lower().strip() for h in highlight_entry]
    else:
        # If it's a string representation of list
        text = str(highlight_entry)
        text = text.strip("[]")
        # Extract items between quotes or brackets
        items = re.findall(r"""['"]([^'"]+)['"]""", text)
        return [i.lower().strip() for i in items if i]

skincare['highlights_list'] = skincare['highlights'].apply(parse_highlights)

In [10]:
skincare.columns

Index(['product_name', 'brand_name', 'loves_count', 'rating', 'reviews',
       'ingredients', 'price_usd', 'highlights', 'secondary_category',
       'tertiary_category', 'size (oz)', 'ings_list', 'count_cleansers',
       'count_moisturizers', 'count_exfoliants', 'count_actives',
       'count_soothing', 'count_preservatives', 'count_fragrance',
       'count_oils', 'count_water_based', 'has_cleansers', 'has_moisturizers',
       'has_exfoliants', 'has_actives', 'has_soothing', 'has_preservatives',
       'has_fragrance', 'has_oils', 'has_water_based', 'highlights_list'],
      dtype='object')

In [11]:
review = pd.read_csv("dataset/review.csv", index_col = 0)

In [12]:
review.head()

,rating,is_recommended,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,review_text,product_name,brand_name,price_usd,skin_light,skin_medium,skin_deep,skin_combination,skin_dry,skin_normal,skin_oily
0,5,1.0,2,0,2,I use this with the Nudestix “Citrus Clean Bal...,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0,0,0,0,0,1,0,0
2,5,1.0,0,0,0,My review title says it all! I get so excited ...,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,1,0,0,0,1,0,0
4,5,1.0,0,0,0,"If you have dry cracked lips, this is a must h...",Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,1,0,0,1,0,0,0
5,4,1.0,1,0,1,The scent isn’t my favourite but it works grea...,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,0,1,0,0,0,1,0
6,2,0.0,8,6,2,I’ll give this 2 stars for nice packaging and ...,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,1,0,0,1,0,0,0


In [13]:
# columns where text mining can be conducted
text_cols = review.columns[review.dtypes == object]
text_cols

Index(['review_text', 'product_name', 'brand_name'], dtype='object')

In [16]:
import re

def basic_clean(s):
    s = str(s).lower()
    s = re.sub(r"\bwon't\b", "will not", s)
    s = re.sub(r"\bcan't\b", "can not", s)
    s = re.sub(r"n't\b", " not", s)
    s = re.sub(r"'re\b", " are", s)
    s = re.sub(r"'s\b", " is", s)
    s = re.sub(r"'d\b", " would", s)
    s = re.sub(r"'ll\b", " will", s)
    s = re.sub(r"'ve\b", " have", s)

    s = re.sub(r"\bsmells\b", "smell", s)
    s = re.sub(r"\bproducts\b", "product", s)

    s = re.sub(r"\bhydrating\b", "hydrate", s)
    s = re.sub(r"\bhydrated\b", "hydrate", s)
    
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    
    return s

text_data = review["review_text"].fillna("").apply(basic_clean)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import numpy as np
import pandas as pd

extra_stop = {
    "like","great","good","just","really","used","using","use","love",
    "feel","feels","feeling","amazing","definitely","recommend",
    "product","skin","face","make","got","one","also","much","well",
    "dont","does","doesn","did","didn","im","ive","youre","weve",
    "received","free","complimentary","testing","purposes","honest",
    "little","nice","ve","don","goes"
}

extra_stop.update({
    "super","time","leaves","cleanser","mask","cream","makeup","night","day",
    "way","makes","long","products","try","sample","works","felt","tried",
    "morning","best","wash","difference","influenster","week","size",
    "absolutely","better","price","bit","never","think","worth","left"
})

all_stopwords = set(ENGLISH_STOP_WORDS).union(extra_stop)

# Keep negation words
for w in ["not", "no", "never"]:
    if w in all_stopwords:
        all_stopwords.remove(w)

all_stopwords.add("not")
all_stopwords = list(all_stopwords)

text_data = review["review_text"].fillna("").apply(basic_clean)

# Vectorizer was used before defined, so declaration swapped to be before utilization
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    min_df=20,
    max_df=0.7,
    stop_words=all_stopwords,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(text_data)

feature_names = np.array(vectorizer.get_feature_names_out())
global_scores = np.asarray(tfidf_matrix.sum(axis=0)).ravel()

mask = np.array([len(t) > 2 for t in feature_names])
feature_names_f = feature_names[mask]
scores_f = global_scores[mask]

top_idx = scores_f.argsort()[::-1][:20]

top_terms = pd.DataFrame({
    "term": feature_names_f[top_idx],
    "score": scores_f[top_idx]
})

top_terms

,term,score
0,dry,4677.232307
1,smell,4279.687857
2,soft,3553.141409
3,hydrate,3464.200900
4,clean,3308.599579
5,sensitive,3059.395978
6,oily,2813.700901
7,smooth,2803.680878
8,moisturizer,2765.920943
9,acne,2561.834960


In [19]:
#bigrams 
vectorizer2 = TfidfVectorizer(
    max_features=3000,
    ngram_range=(2,2),
    min_df=10,
    max_df=0.6,
    stop_words=all_stopwords
)
tfidf2 = vectorizer2.fit_transform(text_data)
terms2 = np.array(vectorizer2.get_feature_names_out())
scores2 = np.asarray(tfidf2.sum(axis=0)).ravel()
top_idx2 = scores2.argsort()[::-1][:20]
pd.DataFrame({"term": terms2[top_idx2], "score": scores2[top_idx2]})

,term,score
0,acne prone,1835.260424
1,fine lines,1104.083796
2,exchange review,981.250564
3,skincare routine,939.291722
4,dry sensitive,907.557276
5,soft smooth,868.397290
6,holy grail,859.366650
7,soft hydrate,702.819032
8,lip balm,692.888077
9,double cleanse,672.851853


In [20]:
# mining reviews to add to the highlights column 

# mine additional highlights from reviews
review_patterns = {
    'acne': r'\bacne\b|\bpimple\b|\bbreakout\b|\bblemish\b',
    'aging': r'\bag(e|ing)\b|\bwrinkle\b|\bfine line\b',
    'pores': r'\bpores?\b|\bclogged\b',
    'redness': r'\bredness\b|\birritat(ed|ion)?\b',
    'dark spots': r'\bdark spots?\b|\bhyperpigmentation\b',
    'texture': r'\btexture\b|\brough\b|\buneven\b',
    'hydrating': r'\bhydrat(ing|ed)?\b|\bmoisturiz(ing|ed)?\b',
    'brightening': r'\bbrighten(ing)?\b|\bglow\b',
    'soothing': r'\bsooth(ing)?\b|\bcalm(ing)?\b',
    'plumping': r'\bplump(ing)?\b',
    'firming': r'\bfirm(ing)?\b|\btight(en)?(ing)?\b'
}

def mine_highlights(review_text):
    if not isinstance(review_text, str) or len(review_text) < 20:
        return []
    
    review_lower = review_text.lower()
    mined = []
    for highlight, pattern in review_patterns.items():
        if re.search(pattern, review_lower):
            mined.append(highlight)
    return mined

review['mined_highlights'] = review['review_text'].apply(mine_highlights)

# create binary columns for your categories
highlight_categories = {
    'concerns': ['acne', 'aging', 'pores', 'redness', 'dark spots', 'texture'],
    'benefits': ['hydrating', 'brightening', 'soothing', 'plumping', 'firming']
}

# Create binary columns
for category, terms in highlight_categories.items():
    for term in terms:
        col_name = f'highlight_{term.replace(" ", "_")}'
        review[col_name] = review['mined_highlights'].apply(
            lambda x: 1 if term in x else 0
        )

In [21]:
review.drop(['mined_highlights', 'review_text'], axis = 1, inplace = True)
skincare.drop(['ingredients', 'highlights', 'ings_list', 'highlights_list'], axis = 1, inplace = True)

In [22]:
print(review.columns[review.dtypes == object])
print(skincare.columns[skincare.dtypes == object])
# Note: we will keep this columns not for the ML model but to display at the end for the product information

Index(['product_name', 'brand_name'], dtype='object')
Index(['product_name', 'brand_name', 'secondary_category',
       'tertiary_category'],
      dtype='object')


In [23]:
print(review.columns)
print(skincare.columns)

Index(['rating', 'is_recommended', 'total_feedback_count',
       'total_neg_feedback_count', 'total_pos_feedback_count', 'product_name',
       'brand_name', 'price_usd', 'skin_light', 'skin_medium', 'skin_deep',
       'skin_combination', 'skin_dry', 'skin_normal', 'skin_oily',
       'highlight_acne', 'highlight_aging', 'highlight_pores',
       'highlight_redness', 'highlight_dark_spots', 'highlight_texture',
       'highlight_hydrating', 'highlight_brightening', 'highlight_soothing',
       'highlight_plumping', 'highlight_firming'],
      dtype='object')
Index(['product_name', 'brand_name', 'loves_count', 'rating', 'reviews',
       'price_usd', 'secondary_category', 'tertiary_category', 'size (oz)',
       'count_cleansers', 'count_moisturizers', 'count_exfoliants',
       'count_actives', 'count_soothing', 'count_preservatives',
       'count_fragrance', 'count_oils', 'count_water_based', 'has_cleansers',
       'has_moisturizers', 'has_exfoliants', 'has_actives', 'has_soothing

In [24]:
# exporting dataset for ML
skincare.to_csv("dataset/skincare_final.csv")
review.to_csv("dataset/review_final.csv")